In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.0):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        # 核心优化1：只用1个Linear生成QKV，然后chunk拆开
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.proj = nn.Linear(d_model, d_model, bias=False)
        self.dropout = dropout

    def forward(self, x, mask=None):
        B, L, _ = x.shape  # [Batch, Seq_len, Dim]
        
        # ----- 1. 投影 + 拆分 (比三个Linear更快，内存连续) -----
        qkv = self.qkv(x)  # [B, L, 3*d_model]
        q, k, v = torch.chunk(qkv, 3, dim=-1)  # 每个 [B, L, d_model]
        
        # ----- 2. 变形为多头 (利用reshape直接展开) -----
        # 目标形状: [B, heads, L, d_k]
        q = q.reshape(B, L, self.num_heads, self.d_k).permute(0, 2, 1, 3)
        k = k.reshape(B, L, self.num_heads, self.d_k).permute(0, 2, 1, 3)
        v = v.reshape(B, L, self.num_heads, self.d_k).permute(0, 2, 1, 3)
        
        # ----- 3. Einsum 注意力 (维度一目了然) -----
        # b: batch, h: heads, i/j: seq_len, d: feature_dim
        attn = torch.einsum('b h i d, b h j d -> b h i j', q, k) / (self.d_k ** 0.5)
        
        # Mask 处理 (广播机制自动适配)
        if mask is not None:
            attn = attn.masked_fill(mask == 0, float('-inf'))
            
        attn = F.softmax(attn, dim=-1)
        attn = F.dropout(attn, p=self.dropout, training=self.training)
        
        # 加权求和: 'b h i j' 对 'b h j d' 做矩阵乘法 -> 'b h i d'
        out = torch.einsum('b h i j, b h j d -> b h i d', attn, v)
        
        # ----- 4. 恢复维度 (permute + reshape) -----
        out = out.permute(0, 2, 1, 3).reshape(B, L, self.d_model)
        
        return self.proj(out)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.0):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        # 依然只用一个 Linear 生成 QKV
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.proj = nn.Linear(d_model, d_model, bias=False)
        self.dropout = dropout

    def forward(self, x, mask=None):
        B, L, _ = x.shape
        
        # 1. 变形为 [B, L, 3, H, d_k]
        qkv = self.qkv(x).reshape(B, L, 3, self.num_heads, self.d_k)
        
        # 2. 妙招：将 '3' 维度转置到 Batch 之后，变成 [3, B, H, L, d_k]
        #    此时 Q, K, V 变成了该张量在 0 维上的切片。
        qkv = qkv.permute(2, 0, 3, 1, 4)  
        
        # 3. 直接索引拆分（零额外开销）
        q, k, v = qkv[0], qkv[1], qkv[2]  # 每个形状: [B, H, L, d_k]
        
        # 4. 缩放点积 (Einsum 展示维度变换)
        attn = torch.einsum('b h i d, b h j d -> b h i j', q, k) / (self.d_k ** 0.5)
        
        if mask is not None:
            attn = attn.masked_fill(mask == 0, float('-inf'))
            
        attn = F.softmax(attn, dim=-1)
        attn = F.dropout(attn, p=self.dropout, training=self.training)
        
        # 5. 加权求和
        out = torch.einsum('b h i j, b h j d -> b h i d', attn, v)
        
        # 6. 合并多头
        out = out.permute(0, 2, 1, 3).reshape(B, L, self.d_model)
        
        return self.proj(out)